# Metamaterial Design with FiberNet

## Closed-Loop Workflow: Unit Cell -> Array -> Simulation -> ML -> RL

This tutorial demonstrates a **professional metamaterial design workflow** using FiberNet v1.25+.

### Design Philosophy

Metamaterials are built from **simple, controllable unit cells** that are:
1. **Tiled** into a 2D array (minimum 3x3)
2. **Welded** at fiber intersections (crosslinks)
3. **Simulated** with physics (Taichi mass-spring / beam FEM)
4. **Optimized** with machine learning and reinforcement learning

```
Unit Cell  --[tile]-->  3x3 Array  --[weld]-->  Complete Graph
    |                                                  |
    v                                                  v
Parameters                                    Simulate (FEM + Dynamics)
    |                                                  |
    +---[RL optimize]<---[ML surrogate]<---[parametric sweep]
```

### API Overview (FiberNet v1.25)

| Function | Purpose |
|----------|---------|
| `fn.create_metamaterial()` | Unit cell -> tile -> weld |
| `fn.simulate_dynamics()` | Taichi mass-spring dynamics |
| `fn.simulate_mechanics()` | Beam FEM mechanics |
| `fn.plot_metamaterial()` | Professional visualization |
| `fn.plot_dynamics()` | Trajectory visualization |
| `fn.analyze()` | Structure analysis |


## 1. Setup & Verification

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import fibernet as fn

print(f"FiberNet v{fn.__version__}")
print(f"Available generators: {len(fn.list_generators())}")


In [ ]:
import os
os.makedirs("fibernet_output/results", exist_ok=True)
os.makedirs("fibernet_output/dataset", exist_ok=True)
os.makedirs("fibernet_output/structures", exist_ok=True)
print("Output directories ready")


## 2. Unit Cell Design

A **unit cell** is the fundamental building block. FiberNet provides parameterized unit cells:

| Unit Cell | Key Parameter | Effect |
|-----------|--------------|--------|
| `reentrant_honeycomb_2d` | `reentrant_angle` | Auxetic when > 90 deg |
| `chiral_honeycomb_2d` | `node_radius` | Rotating node mechanism |
| `star_honeycomb_2d` | `star_angle` | Star-shaped auxetic |
| `arrowhead_auxetic_2d` | `cell_height` | Arrowhead auxetic |


In [ ]:
# Create a single re-entrant honeycomb unit cell
cell = fn.create(
    "reentrant_honeycomb_2d",
    reentrant_angle=150,     # > 90 deg = auxetic geometry
    cell_height=10,          # mm
    cell_width=10,           # mm
    grid_size=(3, 3),        # internal resolution
    radius=0.2,              # fiber radius in mm
)

print("=" * 50)
print("UNIT CELL DATA STRUCTURE")
print("=" * 50)
print(f"Fibers:      {cell.num_fibers}")
print(f"Crosslinks:  {cell.num_crosslinks}")
print(f"Dimension:   {cell.dimension}D")

f0 = cell.fibers[0]
print(f"\nFiber[0]:")
print(f"  fiber_id:    {f0.fiber_id}")
print(f"  length:      {f0.length:.4f} mm")
print(f"  radius:      {f0.radius:.4f} mm")
print(f"  direction:   {f0.direction}")
print(f"  centerline shape: {f0.centerline.shape}")

if cell.crosslinks:
    cl0 = cell.crosslinks[0]
    print(f"\nCrosslink[0]:")
    print(f"  fiber_i: {cl0.fiber_i}, fiber_j: {cl0.fiber_j}")
    print(f"  position: {cl0.position}")
    print(f"  type: {cl0.crosslink_type}")


## 3. Array Construction: Tile + Weld

`fn.create_metamaterial()` performs:
1. Generate unit cell with parameters
2. Tile into Nx x Ny array using `transform.tile()`
3. Auto-detect intersections and weld crosslinks
4. Return complete FiberNetwork with metadata

**Important**: Minimum recommended array size is **3x3**.


In [ ]:
# Create a 3x3 re-entrant honeycomb array
meta = fn.create_metamaterial(
    unit_cell="reentrant_honeycomb_2d",
    array_size=(3, 3),
    weld_threshold=0.5,
    reentrant_angle=150,
    cell_height=10,
    cell_width=10,
    grid_size=(3, 3),
    radius=0.2,
)

fn.print_metamaterial_info(meta)


In [ ]:
# Inspect the mass-spring system (internal data structure)
from fibernet.api import _build_mass_spring_system

edges, rest_lengths, stiffness, masses, positions = _build_mass_spring_system(meta)

print("=" * 60)
print("MASS-SPRING SYSTEM")
print("=" * 60)
print(f"Nodes (crosslinks):  {len(positions)}")
print(f"Springs (segments):  {len(edges)}")
print(f"")
print(f"edges:          shape {edges.shape}")
print(f"  first 5: {edges[:5].tolist()}")
print(f"")
print(f"rest_lengths:   [{rest_lengths.min():.4f}, {rest_lengths.max():.4f}] mm")
print(f"  mean: {rest_lengths.mean():.4f} mm")
print(f"")
print(f"stiffness:      [{stiffness.min():.2e}, {stiffness.max():.2e}] N/m")
print(f"  mean: {stiffness.mean():.2e} N/m")
print(f"")
print(f"masses:         [{masses.min():.6f}, {masses.max():.6f}] kg")
print(f"  total: {masses.sum():.6f} kg")
print(f"")
print(f"positions:      shape {positions.shape}")
print(f"  x: [{positions[:, 0].min():.2f}, {positions[:, 0].max():.2f}]")
print(f"  y: [{positions[:, 1].min():.2f}, {positions[:, 1].max():.2f}]")


## 4. Professional Visualization

In [ ]:
# Plot the metamaterial structure
fig = fn.plot_metamaterial(
    meta,
    show_unit_cells=True,
    show_crosslinks=True,
    colormap="coolwarm",
    save_path="fibernet_output/structures/metamaterial_3x3.png",
)
plt.close()
print("Saved: fibernet_output/structures/metamaterial_3x3.png")


## 5. Taichi Mass-Spring Dynamics

The mass-spring model maps:
- **Crosslinks** -> point masses
- **Fiber segments** -> linear springs with k = E * A / L
- **Stability**: dt < sqrt(m_min / k_max)

```python
traj = fn.simulate_dynamics(network, dt, steps, damping, backend)
```


In [ ]:
# Compute safe time step
m_min = masses.min()
k_max = stiffness.max()
dt_critical = np.sqrt(m_min / k_max)
dt_safe = dt_critical * 0.1

print(f"Stability analysis:")
print(f"  m_min = {m_min:.6e} kg")
print(f"  k_max = {k_max:.2e} N/m")
print(f"  dt_critical = {dt_critical:.2e} s")
print(f"  dt_safe = {dt_safe:.2e} s")


In [ ]:
# Apply tensile load and observe deformation
import numpy as np

# Find boundary nodes
x_coords = positions[:, 0]
y_coords = positions[:, 1]
x_min, x_max = x_coords.min(), x_coords.max()
tol = (x_max - x_min) * 0.05

left_nodes = np.where(x_coords <= x_min + tol)[0].tolist()
right_nodes = np.where(x_coords >= x_max - tol)[0].tolist()

print(f"Boundary conditions:")
print(f"  Fixed nodes (left): {len(left_nodes)}")
print(f"  Loaded nodes (right): {len(right_nodes)}")

# Apply tensile force to right edge
force_per_node = 1e-3  # N (1 mN per node)
total_force = force_per_node * len(right_nodes)
print(f"  Force per node: {force_per_node:.2e} N")
print(f"  Total force: {total_force:.2e} N")

# Compute safe time step
m_min = masses.min()
k_max = stiffness.max()
dt_safe = np.sqrt(m_min / k_max) * 0.05

# External force array: shape (N, 3) - force on each node in x,y,z
n_nodes = len(positions)
ext_force = np.zeros((n_nodes, 3))
for n in right_nodes:
    ext_force[n, 0] = force_per_node  # tension in x-direction

# Run dynamics with loading
print(f"\nRunning dynamics with loading...")
print(f"  dt = {dt_safe:.2e} s")
print(f"  steps = 50000")

traj = fn.simulate_dynamics(
    meta,
    dt=dt_safe,
    steps=50000,
    damping=0.1,
    backend="taichi",
    fixed_nodes=left_nodes,
    external_force=ext_force,
    save_interval=5000,
)

print(f"  Wall time: {traj['time_seconds']:.2f} s")
print(f"  Sim time: {dt_safe * 50000:.2e} s")
print(f"  Final energy: {traj['energy']:.2e} J")
print(f"  Trajectory frames: {len(traj['trajectory'])}")

# Measure deformation
pos_init = traj['initial_positions']
pos_final = traj['positions']
displacements = pos_final - pos_init
max_disp = np.max(np.linalg.norm(displacements, axis=1))
print(f"  Max displacement: {max_disp:.6e} m ({max_disp*1e6:.2f} um)")

# Compute strain
right_disp_x = np.mean(displacements[right_nodes, 0])
original_length = x_max - x_min
strain_measured = right_disp_x / original_length
print(f"  Avg right-edge disp: {right_disp_x:.6e} m")
print(f"  Measured strain: {strain_measured:.4e} ({strain_measured*100:.6f}%)")


In [ ]:
# Visualize dynamics
fig = fn.plot_dynamics(traj, save_path="fibernet_output/results/dynamics.png")
plt.close()
print("Saved: fibernet_output/results/dynamics.png")


## 6. Beam FEM Mechanics

```python
result = fn.simulate_mechanics(network, strain, axis, model)
```


In [ ]:
result = fn.simulate_mechanics(meta, strain=0.001, axis=0, model="linear", segments_per_fiber=3)

print(f"Young's modulus: {result['modulus']:.2e} Pa")
print(f"Strain energy:   {result['energy']:.2e} J")
print(f"Displacements:   {result['displacements'].shape}")


## 7. Parametric Study

Vary unit cell parameters to build ML training dataset.


In [ ]:
import pandas as pd

angles = np.linspace(100, 170, 8)
heights = [8, 10, 12]
radii = [0.15, 0.2, 0.25]

records = []
total = len(angles) * len(heights) * len(radii)
count = 0

for angle in angles:
    for height in heights:
        for radius in radii:
            count += 1
            try:
                m = fn.create_metamaterial(
                    unit_cell="reentrant_honeycomb_2d",
                    array_size=(3, 3),
                    reentrant_angle=angle,
                    cell_height=height,
                    cell_width=10,
                    grid_size=(3, 3),
                    radius=radius,
                )
                res = fn.simulate_mechanics(m, strain=0.001, axis=0, segments_per_fiber=3)
                
                bb_min, bb_max = m.bounding_box()
                size = bb_max - bb_min
                total_length = sum(f.length for f in m.fibers)
                
                records.append({
                    'angle': angle,
                    'cell_height': height,
                    'cell_width': 10.0,
                    'radius': radius,
                    'E_x': res['modulus'],
                    'energy': res['energy'],
                    'num_fibers': m.num_fibers,
                    'num_crosslinks': m.num_crosslinks,
                    'total_length': total_length,
                    'relative_density': total_length * np.pi * radius**2 / (size[0] * size[1]),
                })
                if count % 10 == 0:
                    print(f"  [{count}/{total}] angle={angle:.0f}, h={height}: E={res['modulus']:.2e}")
            except Exception as e:
                print(f"  [{count}/{total}] FAILED: {e}")

df = pd.DataFrame(records)
df.to_csv("fibernet_output/dataset/parametric_study.csv", index=False)
print(f"\nDataset: {len(df)} configurations")


## 8. ML Surrogate Model

Train models to predict E_x from unit cell parameters.


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("fibernet_output/dataset/parametric_study.csv")

# Feature engineering
df['angle_rad'] = np.radians(df['angle'])
df['h_over_w'] = df['cell_height'] / df['cell_width']
df['cos_angle'] = np.cos(df['angle_rad'])
df['sin_angle'] = np.sin(df['angle_rad'])
df['log_E_x'] = np.log10(df['E_x'])

feature_cols = ['angle', 'cell_height', 'cell_width', 'radius',
                'angle_rad', 'h_over_w', 'cos_angle', 'sin_angle']

X = df[feature_cols].values
y_E = df['log_E_x'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset: {len(df)} samples, {len(feature_cols)} features")


In [ ]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}

results_ml = {}
print(f"{'Model':<20} {'CV R2':>8} {'MAE':>8}")
print("-" * 40)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_scaled, y_E, cv=5, scoring='r2')
    model.fit(X_scaled, y_E)
    y_pred = model.predict(X_scaled)
    mae = mean_absolute_error(y_E, y_pred)
    
    results_ml[name] = {
        'model': model,
        'cv_r2_mean': cv_scores.mean(),
        'train_mae': mae,
    }
    print(f"{name:<20} {cv_scores.mean():>8.4f} {mae:>8.4f}")

best_name = max(results_ml, key=lambda k: results_ml[k]['cv_r2_mean'])
best_model = results_ml[best_name]['model']
print(f"\nBest: {best_name} (CV R2 = {results_ml[best_name]['cv_r2_mean']:.4f})")


## 9. Reinforcement Learning

Use ML surrogate as reward function for inverse design.


In [ ]:
class MetamaterialRLAgent:
    def __init__(self, ml_model, scaler, feature_names):
        self.ml_model = ml_model
        self.scaler = scaler
        self.feature_names = feature_names
        self.bounds = {'angle': (100, 170), 'cell_height': (6, 16), 'cell_width': (6, 16), 'radius': (0.1, 0.35)}
        self.param_keys = ['angle', 'cell_height', 'cell_width', 'radius']
        
        from itertools import product
        self.action_space = list(product(*[np.linspace(-1, 1, 5)] * 4))
    
    def _to_features(self, params):
        feat = dict(params)
        feat['angle_rad'] = np.radians(feat['angle'])
        feat['h_over_w'] = feat['cell_height'] / feat['cell_width']
        feat['cos_angle'] = np.cos(feat['angle_rad'])
        feat['sin_angle'] = np.sin(feat['angle_rad'])
        return np.array([[feat[f] for f in self.feature_names]])
    
    def _predict_E(self, params):
        X = self._to_features(params)
        X_s = self.scaler.transform(X)
        return 10 ** self.ml_model.predict(X_s)[0]
    
    def optimize(self, n_episodes=100, n_steps=20, lr=0.1, exploration=0.3):
        q_values = {i: 0.0 for i in range(len(self.action_space))}
        best_reward, best_params = -np.inf, None
        history = []
        
        for ep in range(n_episodes):
            current = {k: np.random.uniform(*self.bounds[k]) for k in self.param_keys}
            ep_reward = 0
            
            for step in range(n_steps):
                a_idx = np.random.randint(len(self.action_space)) if np.random.random() < exploration else max(q_values, key=q_values.get)
                action = np.array(self.action_space[a_idx])
                
                new_params = dict(current)
                for i, key in enumerate(self.param_keys):
                    lo, hi = self.bounds[key]
                    new_params[key] = np.clip(current[key] + action[i] * 0.1 * (hi - lo), lo, hi)
                
                r = np.log10(self._predict_E(new_params))
                q_values[a_idx] += lr * (r - q_values[a_idx])
                ep_reward += r
                
                if r > best_reward:
                    best_reward = r
                    best_params = dict(new_params)
                
                history.append({'episode': ep, 'reward': r, 'best_reward': best_reward, 'E': self._predict_E(new_params), **new_params})
                current = new_params
            
            exploration = max(0.05, exploration * 0.98)
            if (ep + 1) % 25 == 0:
                print(f"  Episode {ep+1}/{n_episodes}: best_reward={best_reward:.3f}")
        
        return history, best_params

rl_agent = MetamaterialRLAgent(best_model, scaler, feature_cols)
print("RL Optimization: 100 episodes x 20 steps")
rl_history, best_rl_params = rl_agent.optimize(n_episodes=100, n_steps=20)

print(f"\nBest parameters:")
for k, v in best_rl_params.items():
    print(f"  {k}: {v:.3f}")
print(f"  Predicted E: {rl_agent._predict_E(best_rl_params):.2e} Pa")


In [ ]:
# Visualize RL results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax = axes[0, 0]
best_rewards = [h['best_reward'] for h in rl_history]
ax.plot(best_rewards, 'r-', linewidth=2)
ax.set_xlabel('Step'); ax.set_ylabel('Best Reward')
ax.set_title('Reward Convergence'); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
E_preds = [h['E'] for h in rl_history]
ax.plot(E_preds, alpha=0.3); ax.set_xlabel('Step'); ax.set_ylabel('E_x (Pa)')
ax.set_title('Predicted Stiffness'); ax.grid(True, alpha=0.3)

ax = axes[1, 0]
angles_hist = [h['angle'] for h in rl_history]
ax.plot(angles_hist, alpha=0.3, color='purple')
ax.axhline(y=best_rl_params['angle'], color='red', linestyle='--', linewidth=2)
ax.set_xlabel('Step'); ax.set_ylabel('Angle (deg)')
ax.set_title('Parameter Trajectory'); ax.grid(True, alpha=0.3)

ax = axes[1, 1]
try:
    best_meta = fn.create_metamaterial(
        unit_cell="reentrant_honeycomb_2d", array_size=(3, 3),
        reentrant_angle=best_rl_params['angle'],
        cell_height=best_rl_params['cell_height'],
        cell_width=best_rl_params['cell_width'],
        radius=best_rl_params['radius'],
    )
    for fiber in best_meta.fibers:
        pts = fiber.centerline[:, :2]
        ax.plot(pts[:, 0], pts[:, 1], 'b-', alpha=0.6, linewidth=1)
    E_best = rl_agent._predict_E(best_rl_params)
    ax.set_title(f"Optimized: E = {E_best:.2e} Pa")
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)
except Exception as e:
    ax.text(0.5, 0.5, f'Error: {e}', ha='center', va='center')

plt.suptitle('RL: Closed-Loop Design', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig("fibernet_output/results/rl_optimization.png", dpi=200, bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: fibernet_output/results/rl_optimization.png")


## 10. FEM Validation

In [ ]:
print("FEM Validation of RL-Optimized Design")
print("=" * 60)

try:
    val_meta = fn.create_metamaterial(
        unit_cell="reentrant_honeycomb_2d", array_size=(3, 3),
        reentrant_angle=best_rl_params['angle'],
        cell_height=best_rl_params['cell_height'],
        cell_width=best_rl_params['cell_width'],
        radius=best_rl_params['radius'],
    )
    
    val_result = fn.simulate_mechanics(val_meta, strain=0.001, axis=0, segments_per_fiber=3)
    E_fem = val_result['modulus']
    E_ml = rl_agent._predict_E(best_rl_params)
    
    print(f"  ML prediction:  E = {E_ml:.2e} Pa")
    print(f"  FEM result:     E = {E_fem:.2e} Pa")
    print(f"  Error:          {abs(E_fem - E_ml) / E_fem * 100:.1f}%")
except Exception as e:
    print(f"  Error: {e}")


## 11. Summary

### Complete Workflow
1. **Unit Cell** - Parameterized building blocks
2. **Array** - 3x3+ tiled arrays with welded crosslinks
3. **Dynamics** - Taichi mass-spring simulation
4. **Mechanics** - Beam FEM structural analysis
5. **Parametric Study** - Dataset generation
6. **ML Surrogate** - Random Forest / GBM / Neural Network
7. **RL Optimization** - Inverse design
8. **FEM Validation** - Confirm with physics

### API Reference
```python
fn.create_metamaterial(unit_cell, array_size, **params)
fn.simulate_dynamics(network, dt, steps, backend)
fn.simulate_mechanics(network, strain, axis)
fn.plot_metamaterial(network)
fn.plot_dynamics(result)
```
